# 01b · 下载 Understat 射门数据（每一脚射门）

**目的**：取得每一脚射门的**位置、身体部位、情境、前一个动作**，用来计算新的“射门风格”指标：

| 想知道的 | 用哪个数据 |
|---|---|
| 头球比例（中锋的空中能力） | 身体部位 |
| 平均射门距离、禁区内比例 | 射门位置 X / Y |
| 直塞球、过人后射门（速度 / 盘带的近似指标） | 前一个动作（**要先确认有没有这一列**） |

⏳ 五个联赛约 **4 万脚射门**，第一次下载可能要 **10-20 分钟**（要打开每一场比赛的页面）。中途断了重新运行就好，已下载的部分会被缓存。

In [1]:
from pathlib import Path
import pandas as pd
import soccerdata as sd

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
print("项目根目录:", ROOT)

[10/23/26 19:12:59] INFO     No custom team name replacements found. You can configure these in       ]8;id=5303209;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=5303210;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\hengk\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=5303216;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=5303217;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\hengk\soccerdata\config\league_dict.json.                                    

项目根目录: c:\Users\hengk\OneDrive\Desktop\Football_Scount


In [12]:
LEAGUES = ["ENG-Premier League", "ESP-La Liga", "GER-Bundesliga", "ITA-Serie A", "FRA-Ligue 1"]
SEASON = "2025/2026"

frames, failed = [], {}
for lg in LEAGUES:
    try:
        us = sd.Understat(leagues=lg, seasons=SEASON)
        shots = us.read_shot_events().reset_index()
        frames.append(shots)
        print(f"✅ {lg:20} {len(shots):>6} 脚射门")
    except Exception as e:
        failed[lg] = repr(e)
        print(f"❌ {lg:20} 失败: {e!r}")
print("\n失败的联赛:", list(failed) or "无")

[10/26/26 12:03:58] INFO     Saving cached data to C:\Users\hengk\soccerdata\data\Understat          ]8;id=5303352;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5303353;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py#300\300]8;;\

✅ ENG-Premier League     9524 脚射门


[10/26/26 12:04:13] INFO     Saving cached data to C:\Users\hengk\soccerdata\data\Understat          ]8;id=5303358;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5303359;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py#300\300]8;;\

✅ ESP-La Liga            9523 脚射门


[10/26/26 12:04:19] INFO     Saving cached data to C:\Users\hengk\soccerdata\data\Understat          ]8;id=5303364;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5303365;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py#300\300]8;;\

✅ GER-Bundesliga         8124 脚射门


[10/26/26 12:06:32] INFO     Saving cached data to C:\Users\hengk\soccerdata\data\Understat          ]8;id=5303370;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5303371;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py#300\300]8;;\

✅ ITA-Serie A            9373 脚射门


[10/26/26 12:06:37] INFO     Saving cached data to C:\Users\hengk\soccerdata\data\Understat          ]8;id=5303376;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=5303377;file://c:\Users\hengk\OneDrive\Desktop\Football_Scount\.venv\Lib\site-packages\soccerdata\_common.py#300\300]8;;\

✅ FRA-Ligue 1            7617 脚射门

失败的联赛: 无


In [13]:
shots = pd.concat(frames, ignore_index=True)
print("形状:", shots.shape)
print("\n列名:", shots.columns.tolist())

# 把文字类的列都列出来，看看有哪些类别（例如身体部位、情境、前一个动作）
for col in shots.columns:
    if shots[col].dtype == "object" and shots[col].nunique() < 40:
        print(f"\n📋 {col}:")
        print(shots[col].value_counts().head(20).to_dict())
display(shots.head())

形状: (44161, 21)

列名: ['league', 'season', 'game', 'team', 'player', 'league_id', 'season_id', 'game_id', 'date', 'shot_id', 'team_id', 'player_id', 'assist_player_id', 'assist_player', 'xg', 'location_x', 'location_y', 'minute', 'body_part', 'situation', 'result']


,league,season,game,team,player,league_id,season_id,game_id,date,shot_id,...,player_id,assist_player_id,assist_player,xg,location_x,location_y,minute,body_part,situation,result
0,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,Bournemouth,Antoine Semenyo,1,2025,28778,2025-08-15 19:00:00,637503,...,11363,8640,Adrien Truffert,0.316921,0.908,0.444,5,Right Foot,Open Play,Missed Shot
1,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,Bournemouth,Antoine Semenyo,1,2025,28778,2025-08-15 19:00:00,637519,...,11363,6820,David Brooks,0.425994,0.908,0.513,63,Left Foot,Open Play,Goal
2,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,Bournemouth,Antoine Semenyo,1,2025,28778,2025-08-15 19:00:00,637523,...,11363,6986,Hamed Junior Traore,0.09559,0.825,0.483,75,Left Foot,Open Play,Goal
3,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,Bournemouth,David Brooks,1,2025,28778,2025-08-15 19:00:00,637509,...,6820,<NA>,<NA>,0.03863,0.855,0.685,41,Right Foot,Open Play,Blocked Shot
4,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,Bournemouth,David Brooks,1,2025,28778,2025-08-15 19:00:00,637521,...,6820,10741,Marcus Tavernier,0.077091,0.896,0.63,70,Left Foot,Open Play,Blocked Shot


In [14]:
out = RAW / "understat_shots_2025_26.csv"
shots.to_csv(out, index=False, encoding="utf-8")
print("✅ 已保存:", out)

✅ 已保存: c:\Users\hengk\OneDrive\Desktop\Football_Scount\data\raw\understat_shots_2025_26.csv
